# **<p align="center"> Title </p>**

## <ins> Description </ins>
### The goal of this notebook is to look at the content of the 4 csv supplementary tables, to understand their content.
<details open>
<summary> <font size="6"> <ins>Key Points 2 </ins> </font> </summary>

- bulletpoint1

-  bulletpoint1

- [link]()

- image1: <img src="" width = 50%>

</details>

</br>

<details open>
<summary> <font size = "6"> <ins> Key Points 2 </ins> </font> </summary>

- `bulletpoint1`


- `bulletpoint2`



</details>

</br>


---

In [ ]:
# Set up variables and imports
from pathlib import Path
root_dir = Path("../../../")
import re
import numpy as np
import pandas as pd
import sys
sys.path.insert(0, root_dir.__str__())
from shutil import copy2, move
import pickle as pkl

import gemmi

from xaidar.data.molecModels import loadPDB, get_pdb_stats, sele_pdb, sele_AA

qfit_dir = root_dir.joinpath("qfit-3.0")
data_dir = root_dir.joinpath("data/entropy/wankowicz")
qfit_pdb_dir = data_dir.joinpath("18209769")
orderParam_dir = data_dir.joinpath("orderParam_analysis")

d:\software\micromamba\envs\qfit\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Bullet point 1

- ### Extract Paper Data

In [ ]:
# Experimental Data
paper_exp_data = data_dir.joinpath("media-1.csv")
paper_exp_df = pd.read_csv(paper_exp_data)
paper_exp_df.head(10)

,Protein,Bound PDB,Apo PDB,dG (kcal/mol),dH (kcal/mol),-TdS (kcal/mol),Temperature,Cell Content,Ligand Content,Cell Concentration,Ligand Concentration,Run Num,Citation (DOI),Paper Ligand Tracker,Notes
0,thermolysin,3T73,1KEI,-7.600382,-3.704589,-3.895793,25,200uL protein,ligand,45 uM protein,1-1.5 mM,summary data,10.1002/cmdc.201200206,TLN-1,info in different paper than ligands 2-9 (corr...
1,thermolysin,3T73,1KEI,-7.538580,-3.603000,-3.935580,25,200uL protein,ligand,30 uM protein,0.4 mM,a,10.1002/cmdc.201200206,TLN-1,NaN
2,thermolysin,3T73,1KEI,-7.675580,-3.740000,-3.935580,25,200uL protein,ligand,30 uM protein,0.4 mM,b,10.1002/cmdc.201200206,TLN-1,NaN
3,thermolysin,3T73,1KEI,-7.613950,-3.738000,-3.875950,25,200uL protein,ligand,30 uM protein,0.4 mM,c,10.1002/cmdc.201200206,TLN-1,NaN
4,thermolysin,3T8F,1KEI,-8.126200,-3.107070,-5.019120,25,200uL protein,ligand,45 uM protein,1-1.5 mM,"1 - Biela et al., 2012 summary",10.1002/cmdc.201200206,TLN-2,measurements in multiple papers
5,thermolysin,3T8F,1KEI,-8.054493,-4.923518,-3.130975,25,200uL protein,ligand,30 uM protein,0.4 mM,"2 - Krimmer et al., 2014 summary",10.1002/cmdc.201400013,TLN-2,NaN
6,thermolysin,3T8F,1KEI,-8.056205,-4.866000,-3.190205,25,200uL protein,ligand,30 uM protein,0.4 mM,a,10.1002/cmdc.201400013,TLN-2,NaN
7,thermolysin,3T8F,1KEI,-8.075575,-4.945000,-3.130575,25,200uL protein,ligand,30 uM protein,0.4 mM,b,10.1002/cmdc.201400013,TLN-2,NaN
8,thermolysin,3T8F,1KEI,-8.018130,-4.977000,-3.041130,25,200uL protein,ligand,30uM protein,0.4 mM,c,10.1002/cmdc.201400013,TLN-2,NaN
9,thermolysin,4MTW,1KEI,-9.058317,-5.114723,-3.943595,25,200uL protein,ligand,30uM protein,0.4mM,summary data,10.1002/cmdc.201400013,TLN-4,NaN


In [ ]:
# Metrics Computed in the Paper
paper_comp_data = data_dir.joinpath("media-2.csv")
paper_comp_df = pd.read_csv(paper_comp_data)
paper_comp_df.head().sort_values("PDB")

,PDB,Protein,Protein_Entropy,dCP,avg_dsasa,avg_dapolar,avg_dpolar,Num_Water_Corrected,Num_Water_Count,TdS,dH,dG
4,7HFK,Mac1,-2.1900,-0.036309,-0.452656,-0.617941,-0.437720,-0.313570,-0.277108,-0.530000,-0.53,-4.350000
2,7HHZ,Mac1,-3.3375,0.156951,-0.639280,-1.196156,-0.342333,-0.132847,-0.096386,-4.973552,-1.20,-6.173552
3,x3450,Mac1,-2.7950,0.130044,-0.314863,-0.630717,-0.075427,-0.168675,-0.205846,-5.450750,-0.96,-6.410750
1,x3458,Mac1,-4.5725,0.425297,0.085791,-0.344632,0.745984,-0.358274,-0.422892,-4.737089,-1.42,-6.157089
0,x4034,Mac1,-4.6075,-0.130274,-0.629385,-0.626116,-0.651253,-0.208117,-0.244578,-5.821826,-1.67,-7.491826


In [ ]:
# Crystallographic metrics of the pdbs used (including resolution, Rfree, etc.)
paper_comp_data = data_dir.joinpath("media-3.csv")
paper_comp_df = pd.read_csv(paper_comp_data)
paper_comp_df.head().sort_values("PDB")

,PDB,Protein,Protein_Entropy,dCP,avg_dsasa,avg_dapolar,avg_dpolar,Num_Water_Corrected,Num_Water_Count,TdS,dH,dG
4,7HFK,Mac1,-2.1900,-0.036309,-0.452656,-0.617941,-0.437720,-0.313570,-0.277108,-0.530000,-0.53,-4.350000
2,7HHZ,Mac1,-3.3375,0.156951,-0.639280,-1.196156,-0.342333,-0.132847,-0.096386,-4.973552,-1.20,-6.173552
3,x3450,Mac1,-2.7950,0.130044,-0.314863,-0.630717,-0.075427,-0.168675,-0.205846,-5.450750,-0.96,-6.410750
1,x3458,Mac1,-4.5725,0.425297,0.085791,-0.344632,0.745984,-0.358274,-0.422892,-4.737089,-1.42,-6.157089
0,x4034,Mac1,-4.6075,-0.130274,-0.629385,-0.626116,-0.651253,-0.208117,-0.244578,-5.821826,-1.67,-7.491826


In [ ]:
# Original source of the PDB files used in the paper 
paper_comp_data = data_dir.joinpath("media-4.csv")
paper_comp_df = pd.read_csv(paper_comp_data)
paper_comp_df.head().sort_values("PDB")

,PDB,Protein,Protein_Entropy,dCP,avg_dsasa,avg_dapolar,avg_dpolar,Num_Water_Corrected,Num_Water_Count,TdS,dH,dG
4,7HFK,Mac1,-2.1900,-0.036309,-0.452656,-0.617941,-0.437720,-0.313570,-0.277108,-0.530000,-0.53,-4.350000
2,7HHZ,Mac1,-3.3375,0.156951,-0.639280,-1.196156,-0.342333,-0.132847,-0.096386,-4.973552,-1.20,-6.173552
3,x3450,Mac1,-2.7950,0.130044,-0.314863,-0.630717,-0.075427,-0.168675,-0.205846,-5.450750,-0.96,-6.410750
1,x3458,Mac1,-4.5725,0.425297,0.085791,-0.344632,0.745984,-0.358274,-0.422892,-4.737089,-1.42,-6.157089
0,x4034,Mac1,-4.6075,-0.130274,-0.629385,-0.626116,-0.651253,-0.208117,-0.244578,-5.821826,-1.67,-7.491826
